![image.png](https://i.imgur.com/a3uAqnb.png)

# Transformer from Scratch for Text Summarization 

In this  assignment, you will implement a **complete Transformer architecture from scratch** for text summarization using the BBC News dataset. This project will deepen your understanding of attention mechanisms, positional encoding, and sequence-to-sequence modeling.

## 🎯 Project Overview
- **Task**: Automatic text summarization of BBC news articles
- **Architecture**: Full Transformer (Encoder-Decoder) implementation from scratch
- **Dataset**: BBC News Summary Dataset
- **Goal**: Build and train a production-ready summarization model

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand and implement multi-head self-attention mechanisms
- Build positional encoding for sequence modeling
- Implement complete Encoder-Decoder Transformer architecture
- Learn advanced training techniques (label smoothing, gradient clipping, learning rate scheduling)
- Evaluate summarization quality using ROUGE metrics
- Master sequence generation and beam search techniques

## 1️⃣ Dataset Download and Initial Exploration

**Task**: Download the BBC News dataset and explore its structure for summarization.

**Requirements**:
- Download BBC News Summary dataset from Kaggle
- Explore the directory structure (articles vs summaries)
- Analyze article and summary length distributions
- Understand the data format and categories

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter
import re

# TODO: Download the BBC News Summary dataset
path = kagglehub.dataset_download("pariza/bbc-news-summary")

def explore_dataset_structure(base_path):
    """
    TODO: Implement function to explore dataset structure
    - Navigate to articles and summaries directories
    - Count files in each category
    - Return directory paths and category lists
    """
    base_path = Path(base_path)
    articles_dir = base_path / "BBC News Summary" / "News Articles"
    summaries_dir = base_path / "BBC News Summary" / "Summaries"

    article_categories = [d.name for d in articles_dir.iterdir() if d.is_dir()]

    total_articles = 0
    for category in article_categories:
        article_files = list((articles_dir / category).glob("*.txt"))
        total_articles += len(article_files)

    print(f"Total articles loaded: {total_articles}")
    return articles_dir, summaries_dir, article_categories

# TODO: Explore dataset structure and display basic statistics
articles_dir, summaries_dir, categories = explore_dataset_structure(path)

# TODO: Load and display sample article and summary to understand data format
sample_category = categories[0]
article_files = list((articles_dir / sample_category).glob("*.txt"))
summary_files = list((summaries_dir / sample_category).glob("*.txt"))

with open(article_files[0], 'r', encoding='utf-8', errors='ignore') as f:
    sample_article = f.read()

sample_summary_file = summaries_dir / sample_category / article_files[0].name
with open(sample_summary_file, 'r', encoding='utf-8', errors='ignore') as f:
    sample_summary = f.read()

print(f"Sample article length: {len(sample_article.split())} words")
print(f"Sample summary length: {len(sample_summary.split())} words")

## 2️⃣ Import Required Libraries

**Task**: Import all necessary libraries for Transformer implementation and training.

**Requirements**:
- Import PyTorch and neural network modules
- Import text processing utilities
- Import evaluation metrics (ROUGE)
- Set up visualization libraries

In [ ]:
# TODO: Install required libraries if needed
# !pip install rouge-score

In [ ]:
# TODO: Import core PyTorch libraries for neural network implementation
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# TODO: Import data processing and utility libraries
import pandas as pd
from pathlib import Path
import re
from tqdm import tqdm
import numpy as np
import pickle
from collections import Counter
import math

## 3️⃣ Data Loading and Text Preprocessing

**Task**: Create a comprehensive data loading pipeline with text cleaning.

**Requirements**:
- Implement BBCNewsDataset class for loading articles and summaries
- Create robust text cleaning function
- Load all categories (politics, entertainment, sport, tech, business)
- Handle encoding errors and empty files
- Display dataset statistics

In [ ]:
class BBCNewsDataset:
    """
    TODO: Implement comprehensive dataset loader for BBC News data
    This class should handle:
    - Loading articles and corresponding summaries
    - Text cleaning and preprocessing
    - Error handling for corrupted files
    """
    def __init__(self, base_path):
        self.base_path = Path(base_path)
        self.articles_dir = self.base_path / "BBC News Summary" / "News Articles"
        self.summaries_dir = self.base_path / "BBC News Summary" / "Summaries"
        self.data = self.load_all_data()

    def clean_text(self, text):
        """
        TODO: Implement text cleaning function
        - Remove special characters and punctuation
        - Convert to lowercase
        - Normalize whitespace
        - Keep only alphanumeric characters and spaces
        """
        # Basic text cleaning
        text = re.sub(r'[^\w\s]', '', text.lower())
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        return text

    def load_all_data(self):
        """
        TODO: Load all articles and summaries from all categories
        - Iterate through all category directories
        - Match articles with their corresponding summaries
        - Apply text cleaning
        - Handle file reading errors gracefully
        """
        data = []
        categories = ['politics', 'entertainment', 'sport', 'tech', 'business']

        for category in categories:
            article_path = self.articles_dir / category
            summary_path = self.summaries_dir / category

            article_files = list(article_path.glob("*.txt"))

            for article_file in article_files:
                try:
                    # TODO: Read article file with proper encoding
                    with open(article_file, 'r', encoding='utf-8', errors='ignore') as f:
                        article_text = f.read()

                    # TODO: Read corresponding summary file
                    summary_file = summary_path / article_file.name
                    with open(summary_file, 'r', encoding='utf-8', errors='ignore') as f:
                        summary_text = f.read()

                    # TODO: Apply text cleaning to both article and summary
                    article_clean = self.clean_text(article_text)
                    summary_clean = self.clean_text(summary_text)

                    # TODO: Filter out empty articles or summaries
                    if len(article_clean.strip()) > 0 and len(summary_clean.strip()) > 0:
                        data.append({
                            'category': category,
                            'filename': article_file.name,
                            'article': article_clean,
                            'summary': summary_clean
                        })
                except Exception as e:
                    print(f"Error processing {article_file}: {e}")
                    continue

        print(f"Loaded {len(data)} article-summary pairs")
        return data

# TODO: Load the complete dataset and convert to DataFrame
bbc_dataset = BBCNewsDataset(path)

# TODO: Convert to pandas DataFrame for easier manipulation
df = pd.DataFrame(bbc_dataset.data)
print(f"Dataset shape: {df.shape}")
print(f"Categories: {df['category'].value_counts()}")

## 4️⃣ Vocabulary Construction

**Task**: Build a comprehensive vocabulary for the Transformer model.

**Requirements**:
- Create Vocabulary class with special tokens (< pad >, < sos >, < eos >, < unk >)
- Implement frequency-based vocabulary filtering
- Handle encoding and decoding of text sequences
- Build vocabulary from both articles and summaries

In [ ]:
class Vocabulary:
    """
    TODO: Implement vocabulary class for text tokenization
    This class should handle:
    - Building vocabulary from text corpus
    - Converting text to token indices
    - Converting indices back to text
    - Handling out-of-vocabulary words
    """
    def __init__(self, min_freq=2):
        # TODO: Initialize special tokens and mappings
        self.min_freq = min_freq
        self.word2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.idx2word = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}
        self.word_freq = Counter()

    def build_vocabulary(self, texts):
        """
        TODO: Build vocabulary from list of texts
        - Count word frequencies across all texts
        - Add words meeting minimum frequency threshold
        - Create bidirectional word-index mappings
        """
        # Count word frequencies
        for text in texts:
            words = text.split()
            self.word_freq.update(words)

        # Add words that meet minimum frequency requirement
        idx = len(self.word2idx)
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq and word not in self.word2idx:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

        print(f"Vocabulary size: {len(self.word2idx)}")

    def encode(self, text, max_length=None):
        """
        TODO: Convert text to sequence of token indices
        - Split text into words
        - Convert words to indices using vocabulary
        - Handle unknown words with <unk> token
        - Add <eos> token and apply length limits
        """
        words = text.split()
        if max_length:
            words = words[:max_length-1]  # Leave space for <eos>

        indices = [self.word2idx.get(word, self.word2idx['<unk>']) for word in words]
        indices.append(self.word2idx['<eos>'])
        return indices

    def decode(self, indices):
        """
        TODO: Convert sequence of indices back to text
        - Map indices to words using vocabulary
        - Remove special tokens for clean output
        - Join words into readable text
        """
        words = [self.idx2word.get(idx, '<unk>') for idx in indices]
        # Remove special tokens for clean output
        words = [w for w in words if w not in ['<pad>', '<sos>', '<eos>']]
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)

# TODO: Build vocabularies from the complete dataset
all_articles = df['article'].tolist()
all_summaries = df['summary'].tolist()

# TODO: Create vocabulary instance and build from both articles and summaries
vocab = Vocabulary(min_freq=2)
vocab.build_vocabulary(all_articles + all_summaries)

## 5️⃣ Sequence-to-Sequence Dataset Preparation

**Task**: Create PyTorch Dataset class for efficient batch processing.

**Requirements**:
- Implement Seq2SeqDataset class for article-summary pairs
- Handle variable-length sequences with proper padding
- Create decoder input and target sequences
- Implement collate function for batch processing
- Split data into training and validation sets

In [ ]:
class Seq2SeqDataset(Dataset):
    """
    TODO: Implement PyTorch Dataset for sequence-to-sequence learning
    This class should handle:
    - Converting text to token sequences
    - Creating decoder input/target pairs
    - Applying sequence length limits
    - Proper tensor conversion
    """
    def __init__(self, articles, summaries, vocab, max_article_len=512, max_summary_len=128):
        self.articles = articles
        self.summaries = summaries
        self.vocab = vocab
        self.max_article_len = max_article_len
        self.max_summary_len = max_summary_len

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        """
        TODO: Return a single training example
        - Encode article and summary using vocabulary
        - Create decoder input (with <sos>) and target (with <eos>)
        - Convert to PyTorch tensors
        """
        article = self.articles[idx]
        summary = self.summaries[idx]

        # TODO: Encode sequences with length limits
        article_encoded = self.vocab.encode(article, self.max_article_len)
        summary_encoded = self.vocab.encode(summary, self.max_summary_len)

        # TODO: Create decoder input and target sequences
        # Decoder input: <sos> + summary[:-1]
        # Decoder target: summary (with <eos>)
        decoder_input = [self.vocab.word2idx['<sos>']] + summary_encoded[:-1]
        decoder_target = summary_encoded

        return {
            'encoder_input': torch.tensor(article_encoded, dtype=torch.long),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }

def collate_fn(batch):
    """
    TODO: Implement collate function for batch processing
    - Pad sequences to same length within batch
    - Use pad_sequence for efficient padding
    - Return dictionary of padded tensors
    """
    encoder_inputs = [item['encoder_input'] for item in batch]
    decoder_inputs = [item['decoder_input'] for item in batch]
    decoder_targets = [item['decoder_target'] for item in batch]

    # TODO: Pad sequences to same length within batch
    encoder_inputs = pad_sequence(encoder_inputs, batch_first=True, padding_value=0)
    decoder_inputs = pad_sequence(decoder_inputs, batch_first=True, padding_value=0)
    decoder_targets = pad_sequence(decoder_targets, batch_first=True, padding_value=0)

    return {
        'encoder_input': encoder_inputs,
        'decoder_input': decoder_inputs,
        'decoder_target': decoder_targets
    }

# TODO: Create train/validation split and datasets
train_size = int(0.9 * len(df))
train_df = df[:train_size]
val_df = df[train_size:]

# TODO: Create training and validation datasets
train_dataset = Seq2SeqDataset(
    train_df['article'].tolist(),
    train_df['summary'].tolist(),
    vocab
)

val_dataset = Seq2SeqDataset(
    val_df['article'].tolist(),
    val_df['summary'].tolist(),
    vocab
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

## 6️⃣ Positional Encoding Implementation

**Task**: Implement sinusoidal positional encoding for the Transformer.

**Requirements**:
- Create PositionalEncoding class using sin/cos functions
- Handle different sequence lengths up to maximum
- Ensure positional encodings are not updated during training
- Implement proper mathematical formulation from "Attention is All You Need"

In [ ]:
class PositionalEncoding(nn.Module):
    """
    TODO: Implement positional encoding using sinusoidal functions
    This module should:
    - Create fixed positional encodings using sin/cos
    - Handle sequences up to max_length
    - Add positional information to input embeddings
    - Use the exact formulation from the Transformer paper
    """
    def __init__(self, d_model, max_length=5000):
        super(PositionalEncoding, self).__init__()

        # TODO: Create positional encoding matrix
        pe = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        
        # TODO: Calculate division term for sinusoidal functions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # TODO: Apply sin to even indices and cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)

        # TODO: Register as buffer so it's not updated during training
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        TODO: Add positional encoding to input embeddings
        Args:
            x: Input embeddings of shape (seq_len, batch_size, d_model)
        Returns:
            x + positional encodings
        """
        return x + self.pe[:x.size(0), :]

## 7️⃣ Transformer Encoder Implementation

**Task**: Implement the Encoder component of the Transformer architecture.

**Requirements**:
- Create Encoder class with embedding and positional encoding
- Use PyTorch's TransformerEncoder layers
- Implement proper masking for padding tokens
- Handle batch-first tensor format
- Apply dropout for regularization

In [ ]:
class Encoder(nn.Module):
    """
    TODO: Implement Transformer Encoder
    This module should include:
    - Word embedding layer
    - Positional encoding
    - Multi-layer Transformer encoder
    - Proper attention masking for padding
    """
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, max_length, dropout=0.1):
        super(Encoder, self).__init__()

        self.d_model = d_model
        
        # TODO: Create embedding layer for input tokens
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # TODO: Add positional encoding
        self.positional_encoding = PositionalEncoding(d_model, max_length)

        # TODO: Create transformer encoder layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Important: use batch_first=True
        )

        # TODO: Stack multiple encoder layers
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):
        """
        TODO: Forward pass through encoder
        Args:
            src: Source sequences (batch_size, seq_len)
            src_mask: Padding mask for source sequences
        Returns:
            Encoded representations (batch_size, seq_len, d_model)
        """
        # TODO: Apply embedding and scale by sqrt(d_model)
        src_emb = self.embedding(src) * math.sqrt(self.d_model)
        
        # TODO: Add positional encoding (requires transpose for positional encoding)
        src_emb = src_emb.transpose(0, 1)  # (seq_len, batch_size, d_model)
        src_emb = self.positional_encoding(src_emb)
        src_emb = src_emb.transpose(0, 1)  # Back to (batch_size, seq_len, d_model)
        src_emb = self.dropout(src_emb)

        # TODO: Create padding mask (True for padding tokens)
        if src_mask is None:
            src_mask = (src == 0)  # True for padding tokens

        # TODO: Pass through transformer encoder with padding mask
        output = self.transformer_encoder(src_emb, src_key_padding_mask=src_mask)
        return output

## 8️⃣ Transformer Decoder Implementation

**Task**: Implement the Decoder component with causal masking and cross-attention.

**Requirements**:
- Create Decoder class with embedding, positional encoding, and output projection
- Implement causal (triangular) masking for autoregressive generation
- Handle cross-attention with encoder outputs
- Create output projection layer for vocabulary predictions
- Implement proper mask generation functions

In [ ]:
class Decoder(nn.Module):
    """
    TODO: Implement Transformer Decoder
    This module should include:
    - Word embedding and positional encoding
    - Multi-layer Transformer decoder with cross-attention
    - Causal masking for autoregressive generation
    - Output projection to vocabulary size
    """
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, max_length, dropout=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        
        # TODO: Create embedding layer for target tokens
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # TODO: Add positional encoding
        self.positional_encoding = PositionalEncoding(d_model, max_length)

        # TODO: Create transformer decoder layer
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        # TODO: Stack multiple decoder layers
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.dropout = nn.Dropout(dropout)
        
        # TODO: Output projection layer to vocabulary
        self.output_projection = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        """
        TODO: Forward pass through decoder
        Args:
            tgt: Target sequences (batch_size, seq_len)
            memory: Encoder outputs (batch_size, src_len, d_model)
            tgt_mask: Causal mask for target sequences
            memory_mask: Padding mask for encoder outputs
        Returns:
            Logits over vocabulary (batch_size, seq_len, vocab_size)
        """
        # TODO: Apply embedding and scale by sqrt(d_model)
        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model)
        
        # TODO: Add positional encoding
        tgt_emb = tgt_emb.transpose(0, 1)  # (seq_len, batch_size, d_model)
        tgt_emb = self.positional_encoding(tgt_emb)
        tgt_emb = tgt_emb.transpose(0, 1)  # Back to (batch_size, seq_len, d_model)
        tgt_emb = self.dropout(tgt_emb)

        # TODO: Create causal mask for target sequence
        if tgt_mask is None:
            seq_len = tgt.size(1)
            tgt_mask = self.generate_square_subsequent_mask(seq_len).to(tgt.device)

        # TODO: Create padding masks
        tgt_key_padding_mask = (tgt == 0)
        
        # Memory padding mask from encoder input
        if memory_mask is None:
            memory_key_padding_mask = None
        else:
            memory_key_padding_mask = memory_mask

        # TODO: Pass through transformer decoder
        output = self.transformer_decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

        # TODO: Project to vocabulary size
        output = self.output_projection(output)
        return output

    def generate_square_subsequent_mask(self, sz):
        """
        TODO: Generate causal mask for autoregressive generation
        Creates upper triangular matrix with -inf values
        """
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask

## 9️⃣ Complete Seq2Seq Transformer Architecture

**Task**: Combine Encoder and Decoder into complete Transformer model.

**Requirements**:
- Create Seq2SeqTransformer class combining encoder and decoder
- Implement forward pass for training
- Add separate encode/decode methods for inference
- Initialize model with appropriate hyperparameters
- Display model size and complexity

In [ ]:
class Seq2SeqTransformer(nn.Module):
    """
    TODO: Implement complete Transformer for sequence-to-sequence learning
    This is the main model class that combines:
    - Transformer Encoder for processing input sequences
    - Transformer Decoder for generating output sequences
    - Methods for both training and inference
    """
    def __init__(self, vocab_size, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, max_length=512, dropout=0.1):
        super(Seq2SeqTransformer, self).__init__()

        # TODO: Initialize encoder component
        self.encoder = Encoder(
            vocab_size, d_model, nhead, num_encoder_layers,
            dim_feedforward, max_length, dropout
        )

        # TODO: Initialize decoder component
        self.decoder = Decoder(
            vocab_size, d_model, nhead, num_decoder_layers,
            dim_feedforward, max_length, dropout
        )

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        TODO: Forward pass for training
        Args:
            src: Source sequences (batch_size, src_len)
            tgt: Target sequences (batch_size, tgt_len)
            src_mask: Source padding mask
            tgt_mask: Target causal mask
        Returns:
            Logits over vocabulary (batch_size, tgt_len, vocab_size)
        """
        # TODO: Encode source sequences
        memory = self.encoder(src, src_mask)
        
        # TODO: Decode target sequences with cross-attention to encoder output
        output = self.decoder(tgt, memory, tgt_mask, src_mask)
        return output

    def encode(self, src, src_mask=None):
        """
        TODO: Encode source sequences (for inference)
        """
        return self.encoder(src, src_mask)

    def decode(self, tgt, memory, tgt_mask=None, memory_mask=None):
        """
        TODO: Decode target sequences given encoder memory (for inference)
        """
        return self.decoder(tgt, memory, tgt_mask, memory_mask)

# TODO: Initialize model with appropriate hyperparameters
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Seq2SeqTransformer(
    vocab_size=len(vocab),
    d_model=768,           # Larger model dimension
    nhead=12,              # More attention heads
    num_encoder_layers=6,   # Standard depth
    num_decoder_layers=6,
    dim_feedforward=3072,   # 4 * d_model
    max_length=512,
    dropout=0.2            # Higher dropout for regularization
).to(device)

# TODO: Display model statistics
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

## 🔟 Training Configuration and Setup

**Task**: Configure training parameters and create data loaders.

**Requirements**:
- Set up data loaders with appropriate batch size
- Configure optimizer (AdamW) with weight decay
- Implement learning rate scheduling (Transformer-style)
- Set up loss function with label smoothing
- Define training hyperparameters

In [ ]:
# TODO: Define training hyperparameters
batch_size = 16          # Adjust based on GPU memory
learning_rate = 2e-4     # Conservative learning rate
num_epochs = 50          # Sufficient for convergence

# TODO: Create data loaders for training and validation
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

# TODO: Set up loss function with label smoothing for better generalization
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

# TODO: Use AdamW optimizer with weight decay
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

def get_lr(step, d_model, warmup_steps=4000):
    """
    TODO: Implement Transformer learning rate schedule
    - Warm up linearly for warmup_steps
    - Then decay proportionally to 1/sqrt(step)
    - Scale by 1/sqrt(d_model)
    """
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## 1️⃣1️⃣ Text Generation Implementation

**Task**: Implement autoregressive text generation for summarization.

**Requirements**:
- Create generate_summary function for inference
- Implement greedy decoding strategy
- Handle sequence termination with < eos > token
- Proper model evaluation mode and no gradient computation
- Clean text output by removing special tokens

In [ ]:
def generate_summary(model, article_text, vocab, device, max_length=128):
    """
    TODO: Generate summary using trained model
    This function should:
    - Encode the input article
    - Generate summary autoregressively
    - Use greedy decoding (argmax)
    - Handle sequence termination
    - Return clean text output
    """
    model.eval()

    with torch.no_grad():
        # TODO: Encode input article
        article_encoded = vocab.encode(article_text, max_length=512)
        src = torch.tensor([article_encoded], dtype=torch.long).to(device)

        # TODO: Get encoder representations
        memory = model.encode(src)

        # TODO: Start generation with <sos> token
        tgt = torch.tensor([[vocab.word2idx['<sos>']]], dtype=torch.long).to(device)

        generated = []

        # TODO: Generate tokens autoregressively
        for _ in range(max_length):
            # TODO: Get next token predictions
            output = model.decode(tgt, memory)

            # TODO: Get the last token logits and apply greedy decoding
            next_token_logits = output[0, -1, :]
            next_token = torch.argmax(next_token_logits).item()

            # TODO: Stop generation if <eos> token is generated
            if next_token == vocab.word2idx['<eos>']:
                break

            generated.append(next_token)

            # TODO: Append generated token to target sequence for next iteration
            next_token_tensor = torch.tensor([[next_token]], dtype=torch.long).to(device)
            tgt = torch.cat([tgt, next_token_tensor], dim=1)

        # TODO: Decode the generated sequence back to text
        summary = vocab.decode(generated)
        return summary

## 1️⃣2️⃣ Training and Validation Functions

**Task**: Implement comprehensive training and validation loops.

**Requirements**:
- Create train_epoch function with gradient clipping
- Implement validate_epoch function for model evaluation
- Include progress tracking with tqdm
- Handle batch processing efficiently
- Apply proper loss calculation and backpropagation

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    TODO: Implement training epoch
    This function should:
    - Set model to training mode
    - Process all training batches
    - Compute loss and gradients
    - Apply gradient clipping
    - Update model parameters
    - Track and return average loss
    """
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(tqdm(train_loader, desc="Training")):
        # TODO: Move batch data to device
        encoder_input = batch['encoder_input'].to(device)
        decoder_input = batch['decoder_input'].to(device)
        decoder_target = batch['decoder_target'].to(device)

        # TODO: Clear gradients
        optimizer.zero_grad()

        # TODO: Forward pass through model
        output = model(encoder_input, decoder_input)

        # TODO: Reshape tensors for loss calculation
        output = output.reshape(-1, output.size(-1))
        decoder_target = decoder_target.reshape(-1)

        # TODO: Calculate loss
        loss = criterion(output, decoder_target)
        
        # TODO: Backward pass
        loss.backward()

        # TODO: Apply gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # TODO: Update parameters
        optimizer.step()
        total_loss += loss.item()

        # TODO: Display progress every 50 batches
        if batch_idx % 50 == 0:
            print(f'Batch {batch_idx}, Loss: {loss.item():.4f}')

    return total_loss / len(train_loader)

def validate_epoch(model, val_loader, criterion, device):
    """
    TODO: Implement validation epoch
    This function should:
    - Set model to evaluation mode
    - Process validation batches without gradients
    - Calculate validation loss
    - Return average validation loss
    """
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            # TODO: Move batch data to device
            encoder_input = batch['encoder_input'].to(device)
            decoder_input = batch['decoder_input'].to(device)
            decoder_target = batch['decoder_target'].to(device)

            # TODO: Forward pass (no gradient computation)
            output = model(encoder_input, decoder_input)

            # TODO: Reshape and calculate loss
            output = output.reshape(-1, output.size(-1))
            decoder_target = decoder_target.reshape(-1)

            loss = criterion(output, decoder_target)
            total_loss += loss.item()

    return total_loss / len(val_loader)

## 1️⃣3️⃣ ROUGE Score Evaluation

**Task**: Implement ROUGE metrics for evaluating summarization quality.

**Requirements**:
- Install and import rouge-score library
- Create function to calculate ROUGE-1, ROUGE-2, and ROUGE-L scores
- Evaluate on subset of validation data for efficiency
- Compare generated summaries with ground truth
- Display comprehensive evaluation metrics

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

def calculate_rouge_scores(model, dataset, vocab, device, num_samples=50):
    """
    TODO: Calculate ROUGE scores for model evaluation
    This function should:
    - Use rouge_scorer for standard ROUGE metrics
    - Generate summaries for a sample of validation data
    - Compare with ground truth summaries
    - Calculate ROUGE-1, ROUGE-2, and ROUGE-L scores
    - Return average scores across all samples
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    # TODO: Select random subset for evaluation efficiency
    subset_size = min(num_samples, len(dataset))
    indices = np.random.choice(len(dataset), subset_size, replace=False)

    print(f"Calculating ROUGE scores on {subset_size} samples...")

    for i, idx in enumerate(tqdm(indices)):
        # TODO: Get article and reference summary
        article = dataset.articles[idx]
        reference_summary = dataset.summaries[idx]

        # TODO: Generate summary using the model
        generated_summary = generate_summary(model, article, vocab, device)

        # TODO: Calculate ROUGE scores between generated and reference
        scores = scorer.score(reference_summary, generated_summary)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    # TODO: Calculate and display average scores
    avg_rouge1 = np.mean(rouge1_scores)
    avg_rouge2 = np.mean(rouge2_scores)
    avg_rougeL = np.mean(rougeL_scores)

    print(f"\n{'='*50}")
    print("ROUGE SCORES SUMMARY")
    print(f"{'='*50}")
    print(f"ROUGE-1: {avg_rouge1:.4f}")
    print(f"ROUGE-2: {avg_rouge2:.4f}")
    print(f"ROUGE-L: {avg_rougeL:.4f}")

    return {
        'rouge1': avg_rouge1,
        'rouge2': avg_rouge2,
        'rougeL': avg_rougeL
    }

## 1️⃣4️⃣ Pre-training Model Evaluation

**Task**: Evaluate the untrained model to establish baseline performance.

**Requirements**:
- Calculate ROUGE scores before training
- Test generation capability on sample articles
- Display baseline performance metrics
- Compare generated summaries with ground truth

In [ ]:
# TODO: Evaluate untrained model to establish baseline
rouge_scores = calculate_rouge_scores(model, val_dataset, vocab, device, num_samples=50)

In [ ]:
# TODO: Test the untrained model on sample articles
print("Testing the untrained model:")
print("=" * 80)

for i in range(3):
    sample = df.iloc[i]
    article = sample['article']
    ground_truth = sample['summary']
    
    # TODO: Generate summary with untrained model
    generated = generate_summary(model, article, vocab, device)

    print(f"\nSample {i+1} - Category: {sample['category']}")
    print(f"Article: {article}...")
    print(f"Ground Truth: {ground_truth}")
    print(f"Generated: {generated}")
    print("-" * 80)

## 1️⃣5️⃣ Complete Training Loop

**Task**: Implement the main training loop with learning rate scheduling and progress tracking.

**Requirements**:
- Implement full training loop for specified epochs
- Apply Transformer learning rate scheduling
- Track training and validation losses
- Save model checkpoints
- Display detailed training progress

In [ ]:
# TODO: Initialize training tracking variables
train_losses = []
val_losses = []
step = 0

# TODO: Main training loop
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 30)
    
    # TODO: Set model to training mode
    model.train()
    total_loss = 0
    
    # TODO: Training phase with learning rate scheduling
    for batch_idx, batch in enumerate(tqdm(train_loader, desc="Training")):
        step += 1
        
        # TODO: Update learning rate using Transformer schedule
        lr = get_lr(step, 768)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        
        # TODO: Move batch to device
        encoder_input = batch['encoder_input'].to(device)
        decoder_input = batch['decoder_input'].to(device)
        decoder_target = batch['decoder_target'].to(device)

        # TODO: Forward pass
        optimizer.zero_grad()
        output = model(encoder_input, decoder_input)
        
        # TODO: Calculate loss
        output = output.reshape(-1, output.size(-1))
        decoder_target = decoder_target.reshape(-1)
        
        loss = criterion(output, decoder_target)
        
        # TODO: Backward pass with gradient clipping
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        
        total_loss += loss.item()

        # TODO: Display progress every 50 batches
        if batch_idx % 50 == 0:
            print(f'Batch {batch_idx}, Loss: {loss.item():.4f}, LR: {lr:.6f}')

    # TODO: Calculate average training loss
    train_loss = total_loss / len(train_loader)
    train_losses.append(train_loss)
    
    # TODO: Validation phase
    val_loss = validate_epoch(model, val_loader, criterion, device)
    val_losses.append(val_loss)

    # TODO: Display epoch results
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

# TODO: Save trained model and training history
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'train_losses': train_losses,
    'val_losses': val_losses
}, 'seq2seq_model.pth')

## 1️⃣6️⃣ Post-training Model Evaluation

**Task**: Evaluate the trained model and compare with baseline performance.

**Requirements**:
- Calculate ROUGE scores after training
- Compare improvement from baseline
- Analyze model performance on validation set
- Display comprehensive evaluation metrics

In [ ]:
# TODO: Evaluate trained model performance
rouge_scores = calculate_rouge_scores(model, val_dataset, vocab, device, num_samples=50)

## 1️⃣7️⃣ Sample Generation

**Task**: Test the trained model on various examples and analyze generation quality.

**Requirements**:
- Test on sample articles
- Compare generated summaries with ground truth
- Analyze improvement in generation quality
- Test on custom articles

In [ ]:
# TODO: Test the trained model on sample articles
print("Testing the trained model:")
print("=" * 80)

for i in range(3):
    sample = df.iloc[i]
    article = sample['article']
    ground_truth = sample['summary']
    
    # TODO: Generate summary with trained model
    generated = generate_summary(model, article, vocab, device)

    print(f"\nSample {i+1} - Category: {sample['category']}")
    print(f"Article: {article}...")
    print(f"Ground Truth: {ground_truth}")
    print(f"Generated: {generated}")
    print("-" * 80)

In [ ]:
# TODO: Test with custom article to evaluate generalization
custom_article = """
the government announced new policies to tackle climate change today
the prime minister said that renewable energy investments will increase by fifty percent next year
solar and wind power projects will receive additional funding
environmental groups welcomed the announcement but said more action is needed
"""

print("\nCustom Article Test:")
print(f"Article: {custom_article.strip()}")
print(f"Generated Summary: {generate_summary(model, custom_article, vocab, device)}")

## 1️⃣8️⃣ Training Results Visualization and Analysis

**Task**: Create comprehensive visualizations of training progress and final performance.

**Requirements**:
- Plot training and validation loss curves
- Analyze convergence and overfitting
- Display final performance metrics
- Provide insights into model behavior

In [ ]:
# TODO: Create comprehensive training visualization
plt.figure(figsize=(12, 4))

# TODO: Plot training and validation losses
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'bo-', label='Train')
plt.plot(epochs, val_losses, 'ro-', label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss per Epoch')
plt.legend()

plt.tight_layout()
plt.show()

# TODO: Display final training statistics
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final validation loss: {val_losses[-1]:.4f}")

### 📊 Evaluation Criteria

Your assignment will be evaluated based on:

1. **Architecture Implementation (50%)**
   - Correct Positional Encoding implementation
   - Proper Transformer Encoder with multi-head attention
   - Correct Transformer Decoder with causal masking
   - Appropriate text generation function
   - Proper handling of padding and special tokens

2. **Training Implementation (30%)**
   - Correct dataset preparation and batching
   - Proper loss calculation with label smoothing
   - Effective training loop with gradient clipping
   - Learning rate scheduling implementation
   - Model convergence and validation

3. **Code Quality and Analysis (20%)**
   - Clean, well-documented code with proper structure
   - Comprehensive ROUGE evaluation
